# 0.94650, forkable — the top recipe rebuilt from public parts only

The notebook at the top of the Code tab,
[@taeyangg4's lexsort master](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master),
reads `sub_053_quad_paradigm_master.csv` from a dataset that is not public. Fork it and that cell fails.
Its recipe is published, though, and the private file sits in a 6% slot:

```
blend = 0.90 * lexrank(anchor, RealMLP) + 0.06 * rank(private stack) + 0.04 * rank(RealMLP)
blend += the four boundary shifts;   final = lexrank(blend, RealMLP)
```

This notebook fills that 6% slot with an ensemble of my own, built from out-of-fold predictions, and
replaces the public RealMLP with the same architecture retrained on my folds and averaged over three
seeds. Every input here is public. **It scores 0.94650**, the same as the original.

| slot | original | here | public LB alone |
|---|---|---|---|
| 90% anchor | [@jazivxt's](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) `submission_latest_best.csv` | unchanged | 0.94649 |
| 6% stack | a private file | six tree views + RealMLP + [@najiama's XGBoost](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639), weighted on OOF | 0.94645 |
| 4% + tie-break | [@yekenot's RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) | the same model retrained on my 10 folds, 3 seeds | OOF 0.946182 vs 0.946010 |

| variant | public LB |
|---|---|
| original notebook | 0.94650 |
| **this rebuild** | **0.94650** |
| rebuild with the 6% slot raised to 11% | 0.94650 |
| rebuild with the anchor replaced by the average of jazivxt's two 0.94649 files | 0.94649 |
| the 6% stack submitted on its own | 0.94645 |

Three things I measured on the way, all negative, so you can skip them.

### 1. The tie-breaking cannot move the score

The anchor has 16,846 repeated values, and resolving them is the original notebook's headline. The tie
groups are 15,362 pairs, 712 triples, 19 larger. That is about 5,900 positive-negative pairs inside
groups, and a tied pair scores 0.5 instead of 0 or 1, so **a perfect tie-breaker is worth at most
2.5e-7** — a fortieth of the last displayed digit. The cell below counts it on the actual file.

### 2. There are no more deterministic cells to find

The four boundary rules are real: pure on all 668,665 training rows and worth +0.6e-5 on honest OOF.
Looking for more is where it goes wrong. Keeping every cell whose training labels happen to be
identical finds 12,000 of them and costs **-0.0024**, because at a 17.5% base rate a run of 50 zeros is
common. Filtering by significance instead — keep a cell only if the model expects at least 12 buyers in
it and the data has none — finds **zero** cells. The reason is in the calibration: this ensemble's
probability at the 5th percentile is 0.00007, so it already treats those rows as impossible.

### 3. Seed averaging pays on the board and is invisible in OOF

Three RealMLP seeds instead of one moved the stack's OOF by +0.000001 and its public LB by +0.00001. An
out-of-fold row is predicted by one fold model; a test row is the average of ten. Seed noise cancels on
test in a way your OOF cannot show you. Tune weights on OOF, then average seeds for the submission.

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"{ROOT}/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

rk = lambda v: rankdata(v, method="average") / len(v)
test = pd.read_csv(find_one("playground-series-s6e9/test.csv", "test.csv")).sort_values(ID).reset_index(drop=True)
ids = test[ID].to_numpy(); N = len(test)

def load(*pats, col=None):
    d = pd.read_csv(find_one(*pats)).sort_values(ID)
    assert np.array_equal(d[ID].to_numpy(), ids)
    return d[col or [c for c in d.columns if c != ID][0]].to_numpy(dtype=float)

anchor = load("submission_latest_best.csv")                       # jazivxt, 0.94649
six = pd.read_csv(find_one("test_six_views.csv")).sort_values(ID).reset_index(drop=True)
nn = load("test_realmlp_g.csv", col="G_realmlp_3seed")            # my RealMLP, 10 folds, 3 seeds
xgb = load("xgboost-triple-te-dynamic-pruning-lb-0-94639/submission_XGBOOST.csv", "submission_XGBOOST.csv")

VIEWS = {"A_lgbm_triple_te_digits_3seed": .2, "B_xgb_on_A_features": .2, "C_no_digits_windows_lift_sm2_30_300": .1,
         "D_no_exact_key_ladder_windows": .2, "E_ladder25_250_2500_lift_sm5_50_500": .1, "F_exact_rate_as_init_score": .2}
trees = rk(sum(w * rk(six[c].to_numpy(dtype=float)) for c, w in VIEWS.items()))
stack = rk(0.50 * trees + 0.25 * rk(nn) + 0.25 * rk(xgb))         # weights hill-climbed on OOF, nested +5.5e-5
print(f"anchor ties: {N - len(np.unique(anchor)):,}")
print(f"spearman(stack, anchor) = {spearmanr(stack, anchor).statistic:.5f}")

## What the ties are actually worth

In [ ]:
vc = pd.Series(anchor).value_counts(); g = vc[vc > 1]
n = g.to_numpy(); p = np.array(g.index, dtype=float)
mixed = float((n * (n - 1) / 2 * 2 * p * (1 - p)).sum())          # expected pos-neg pairs inside tie groups
n1 = 0.1746 * N; n0 = N - n1
print(f"tie groups {len(g):,}  sizes {dict(pd.Series(n).value_counts().sort_index().head(4))}")
print(f"expected positive-negative pairs inside groups: {mixed:,.0f}")
print(f"most a perfect tie-breaker can add: {0.5 * mixed / (n1 * n0):.2e} AUC")

## The recipe

In [ ]:
def lexrank(primary, secondary):
    o = np.lexsort((secondary, primary))
    r = np.empty(N); r[o] = np.arange(1, N + 1)
    return (r - 0.5) / N

inc = test.Annual_Income_USD.to_numpy(float); km = test.Daily_Commute_km.to_numpy(float)
env1 = test.Environmental_Concern_Level.to_numpy() == 1
no_sub = test.Subsidy_Available.astype(str).to_numpy() == "No"
anx_mh = test.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()
shift = np.zeros(N)
shift[inc >= 170537] += 10                       # 393/393 buyers in train
shift[(inc >= 31004) & (inc <= 41970)] -= 10     # 0/1,257
shift[km >= 83] -= 5                             # 0/186
shift[(inc == 30000) & no_sub & (env1 | anx_mh)] -= 5   # 0/7,157
print("rows shifted:", int((shift != 0).sum()))

blend = 0.90 * lexrank(anchor, nn) + 0.06 * rk(stack) + 0.04 * rk(nn)
final = lexrank(blend + shift, nn)
assert len(np.unique(final)) == N and np.isfinite(final).all()
pd.DataFrame({ID: ids, TARGET: final}).to_csv("submission.csv", index=False)
print(f"wrote submission.csv | spearman vs the anchor {spearmanr(final, anchor).statistic:.6f}")

## Where the ceiling is

Nine tenths of this file is one anchor that scores 0.94649 by itself, so everything else is fighting
over the last ten percent. I swept my stack's slot from 6% to 30% and the board reads 0.94650, 0.94650,
0.94649, 0.94648 — flat, then down. The stack is 0.00005 behind the anchor and diversity only pays near
parity, which is also why it cost a digit when I mixed it on top of the finished file.

What would move it is an independent model at 0.94650 on its own. My OOF-only ensemble reached 0.94645
without touching a public blend ([notebook](https://www.kaggle.com/code/megayak/s6e9-what-the-0-94650-actually-is)
has the full ladder), and the member that did the most for it was the neural network — the only strong
model in the pool that is not a tree. A second one of those is the thing I have not found.

---
Credit: [@taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) for the
recipe and the four rules, [@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) for the
anchor, [@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) for RealMLP,
[@najiama](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639) for the XGBoost and
its OOF. The tree views, the RealMLP retrain, the OOF weighting and the measurements are mine
([library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library)).
If a forkable 0.94650 is useful to you, an upvote helps.